<a href="https://colab.research.google.com/github/simongalaxy/sentimental_agent/blob/master/Amazon_Reviews_Data_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#! pip install sentence-transformers transformers

In [ ]:
import pandas as pd
from typing import List, Dict
from pprint import pformat

In [ ]:
filePath = "/content/amazon_google_reviews.csv"

In [ ]:
def load_clean_split_data_from_csv(path: str) -> List[pd.DataFrame]:

  # load data from csv file.
  df = pd.read_csv(path)
  print(df.info())

  # clean data.
  str_cols = ["Platform", "Product/Location_Name", "Review_Text"]
  df[str_cols] = df[str_cols].apply(lambda x: x.str.strip())

  # split data into a list of dataframes by platforms and products.
  dfs = []
  for platform in df["Platform"].unique():
    for product in df[df["Platform"] == platform]["Product/Location_Name"].unique():
      df_temp = df[(df["Platform"] == platform) & (df["Product/Location_Name"] == product)].reset_index()
      dfs.append(df_temp)

  print(f"Total {len(dfs)} dataframes splitted.")

  return dfs

In [ ]:
dfs = load_clean_split_data_from_csv(path = filePath)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6200 entries, 0 to 6199
Data columns (total 7 columns):
 #   Column                 Non-Null Count  Dtype 
---  ------                 --------------  ----- 
 0   Review_ID              6200 non-null   object
 1   Platform               6200 non-null   object
 2   Product/Location_Name  6200 non-null   object
 3   Review_Text            6200 non-null   object
 4   Rating                 6200 non-null   int64 
 5   Date                   6200 non-null   object
 6   Helpful_Votes          6200 non-null   int64 
dtypes: int64(2), object(5)
memory usage: 339.2+ KB
None
Total 25 dataframes splitted.


In [ ]:
from sentence_transformers import SentenceTransformer
from sentence_transformers.util import paraphrase_mining

# --- Assumed Setup ---
# df = pd.DataFrame({"Review_Text": [...]})

# 2. Load a lightweight, high-performance embedding model
model = SentenceTransformer("all-MiniLM-L6-v2")

def group_semantic_similar_reviews(df: pd.DataFrame, model) -> pd.DataFrame:
  # 1. Extract reviews to a list
  sentences = df["Review_Text"].tolist()

  # 3. Perform Paraphrase Mining
  # (Removed the invalid 'min_score' argument)
  pairs = paraphrase_mining(model, sentences)

  # 4. Group the sentences using a Union-Find (Connected Components) structure
  parent = {i: i for i in range(len(sentences))}

  def find(i):
      if parent[i] == i:
          return i
      parent[i] = find(parent[i])  # Path compression
      return parent[i]

  def union(i, j):
      root_i = find(i)
      root_j = find(j)
      if root_i != root_j:
          parent[root_j] = root_i

  # Merge sentences ONLY if they meet or exceed the similarity threshold
  threshold = 0.85
  for score, i, j in pairs:
      if score >= threshold:
          union(i, j)

  # 5. Map the cluster root IDs directly back to your DataFrame
  # This gives every review in df a Group ID based on its semantic cluster root
  df["Semantic_Group_ID"] = [find(i) for i in range(len(sentences))]

  # 6. Extract the groupings for visual confirmation
  groupings = {}
  for i in range(len(sentences)):
      root = find(i)
      if root not in groupings:
          groupings[root] = []
      groupings[root].append(sentences[i])

  # 7. Print the Results
  print("--- SEMANTIC GROUPINGS ---")
  deduplicated_sentences = []

  for group_id, group in groupings.items():
      print(f"\nGroup {group_id}:")
      for sentence in group:
          print(f" - {sentence}")

      # The representative sentence of this group is the first one
      deduplicated_sentences.append(group[0])

  print("\n--- FINAL DEDUPLICATED LIST ---")
  for sent in deduplicated_sentences:
      print(f"✔ {sent}")

  return df


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [ ]:
dfs_processed = [group_semantic_similar_reviews(df=df, model=model) for df in dfs]

串流輸出內容已截斷至最後 5000 行。
Group 60:
 - Honestly, absolute perfection! easily the best purchase i have made this year. high quality materials, elegant design, and works flawlessly.
 - Absolute perfection! Easily the best purchase I have made this year. High quality materials, elegant design, and works flawlessly. #verifiedpurchase
 - Summary: Absolute perfection! Easily the best purchase I have made this year. High quality materials, elegant design, and works flawlessly.
 - Bought this recently. Absolute perfection! Easily the best purchase I have made this year. High quality materials, elegant design, and works flawlessly.
 - Honestly, absolute perfection! easily the best purchase i have made this year. high quality materials, elegant design, and works flawlessly.
 - Absolute perfection! Easily the best purchase I have made this year. High quality materials, elegant design, and works flawlessly. #verifiedpurchase
 - Absolute perfection! Easily the best purchase I have made this year. High q

In [ ]:
def consolidate_reviews_by_groups(df: pd.DataFrame) -> pd.DataFrame:
  # consolidate the df by Semantic_Group_ID.
  df_grouped = df.groupby(by=["Platform", "Product/Location_Name", "Semantic_Group_ID"]).agg(Count=("Review_ID", "count"), Average_Rating=("Rating", "mean"), Helpful_Votes=("Helpful_Votes", "sum")).reset_index()

  # get the first sentences by Semantic_Group_IDs.
  dfs_first_sentence = []
  for id in df["Semantic_Group_ID"].unique():
    temp = df[df["Semantic_Group_ID"] == id][["Semantic_Group_ID", "Review_Text"]].head(1)
    dfs_first_sentence.append(temp)
  df_first_sentence = pd.concat(dfs_first_sentence)

  # merge these 2 dataframes as output.
  df_unique_review = pd.merge(df_grouped, df_first_sentence, on="Semantic_Group_ID")

  return df_unique_review

In [ ]:
dfs_unique_review = [consolidate_reviews_by_groups(df=df) for df in dfs_processed]

In [ ]:
for df in dfs_unique_review:
  platform = df["Platform"].to_list()[0].replace(" ", "_")
  product = df["Product/Location_Name"].to_list()[0].replace(" ", "_")
  filename = f"{platform}-{product}-unique_review.csv"
  df[["Platform", "Product/Location_Name", "Semantic_Group_ID", "Review_Text", "Count", "Average_Rating", "Helpful_Votes"]].sort_values(by="Review_Text", ascending=True).to_csv(filename)